
# NIFTY Volatility Shocks: 15-Minute Intraday GARCH(1,1)

## Objective
We will fit a standard GARCH(1,1) model to the **15-minute intraday log returns** of the NIFTY index to predict **Expected 15m Volatility**. We will compare this expectation to the **Actual 15m Realized Volatility** (calculated from high-frequency 1-minute data). 

A "Shock" is defined as a 15-minute interval where the Actual Volatility exceeds the Forecasted Volatility by more than 2 Standard Deviations. We will analyze how the frequency of these intraday shocks changed across the three regulatory regimes.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11


### Methodology & Financial Significance

**What exactly are we doing here?**
1. **Intraday Granularity:** We broke down the NIFTY index into 15-minute intervals. Instead of looking at whether a *whole day* was volatile, we are zooming in to see if specific 15-minute windows experienced unnatural bursts of volatility.
2. **The "Baseline" (GARCH):** We use a GARCH(1,1) model on these 15-minute returns. GARCH stands for *Generalized Autoregressive Conditional Heteroskedasticity*. In simple terms, it calculates the "expected" volatility for the next 15 minutes based on recent history. If the market has been quiet all morning, the GARCH model expects the next 15 minutes to also be quiet.
3. **The "Shock" (Realized vs Expected):** We measure the *actual* realized volatility during that 15-minute window using high-frequency 1-minute data. We then compare the Actual Volatility against the GARCH Expected Volatility.
4. **The 2-Sigma Threshold:** If the Actual Volatility is higher than the Expected Volatility by more than 2 Standard Deviations, we flag it as a **Shock**. Statistically, a positive 2-sigma deviation should be a rare outlier event in a normal market.

**What does this signify?**
In a healthy, mature market, volatility tends to cluster (quiet periods stay quiet, volatile periods stay volatile). Our GARCH model perfectly maps this natural clustering. 

However, during the "0DTE Frenzy" (Phase 2), market dynamics changed. Huge waves of speculative retail options trading—especially gamma squeezes and stop-loss hunting right before expiry—caused sudden, explosive spikes in volatility that mathematical models simply couldn't anticipate. 

By tracking the frequency of these "Shocks" across the three regulatory regimes, we are quantitatively measuring **how much unnatural, unpredictable chaos the 0DTE speculation injected into the intraday microstructure**, and whether the SEBI mandate successfully suppressed it.


## 1. Data Ingestion & Preprocessing

1. **High-Frequency Data**: We load `nifty.csv` (1-minute bars) to compute 15-Minute Realized Volatility.
2. **15-Minute Returns**: We resample the 1-minute data to 15-minute bars. To prevent massive overnight gaps from destroying the GARCH model, the return for the very first bar of the day (09:15) is calculated strictly intraday (Open to Close), while all subsequent bars use standard Close-to-Close returns.


In [ ]:

# Load 1-minute data
print("Loading 1-minute NIFTY dataset...")
df = pd.read_csv('../data/processed/nifty.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# 1. Calculate 15-Minute Realized Volatility (Actual Volatility)
# Compute 1-minute intraday log returns (strictly within the day)
df['log_ret_1m'] = df.groupby(df.index.date)['close'].transform(lambda s: np.log(s / s.shift(1)))
df['log_ret_1m'] = df['log_ret_1m'].fillna(0)

# Resample to 15-minute bins and sum squared returns
rv_15m = df['log_ret_1m'].pow(2).resample('15min').sum()

# Annualize: 25 bars of 15m per 375m trading day. 25 * 252 = 6300 periods per year
actual_vol_ann = np.sqrt(rv_15m * 25 * 252)
actual_vol_ann = actual_vol_ann[actual_vol_ann > 0] # Filter empty overnight bins

# 2. Prepare 15-Minute Data for GARCH
df_15m = df.resample('15min').agg({'open': 'first', 'close': 'last'}).dropna()

# Standard return: close to previous close
df_15m['log_ret_15m'] = np.log(df_15m['close'] / df_15m['close'].shift(1)) * 100

# Fix the overnight gap for the first bar of the day (09:15)
# Use Open to Close return instead of Previous Close to Close
is_first_bar = df_15m.index.strftime('%H:%M') == '09:15'
df_15m.loc[is_first_bar, 'log_ret_15m'] = np.log(df_15m.loc[is_first_bar, 'close'] / df_15m.loc[is_first_bar, 'open']) * 100

# Merge RV into df_15m
df_15m['actual_vol'] = actual_vol_ann
df_15m = df_15m.dropna(subset=['log_ret_15m', 'actual_vol']).copy()

# Add date column for later grouping
df_15m['date'] = df_15m.index.normalize()

print(f"Prepared {len(df_15m)} 15-minute intraday intervals.")



## 2. GARCH(1,1) Volatility Forecasting

We fit a GARCH(1,1) model on the continuous stream of 15-minute log returns.
The model estimates the conditional variance for each 15-minute period, which we then annualize (multiply by sqrt(6300)) to serve as our **Forecasted Volatility**.


In [ ]:

# Fit GARCH(1,1) model on 15m returns
# rescaling y avoids convergence warnings
garch = arch_model(df_15m['log_ret_15m'], mean='Zero', vol='Garch', p=1, q=1, dist='Normal', rescale=True)
res = garch.fit(disp='off')

print(res.summary())

# Extract conditional volatility (percentage scale) and annualize it
# Since data is 15-minute, annualization factor is sqrt(25 * 252) = sqrt(6300)
# (rescale=True means conditional_volatility is scaled by the rescale factor, so we divide by it)
df_15m['forecast_vol'] = (res.conditional_volatility / res.scale) / 100 * np.sqrt(25 * 252)

display(df_15m[['actual_vol', 'forecast_vol']].head())



## 3. Identifying Volatility Shocks

We calculate the forecast error: `Actual Volatility - Forecasted Volatility`.
A "Positive Shock" is flagged when the actual 15-minute volatility is surprisingly higher than the forecast by more than 2 standard deviations of the error.


In [ ]:

# Calculate Forecast Error
df_15m['forecast_error'] = df_15m['actual_vol'] - df_15m['forecast_vol']

# Calculate Standard Deviation of the error
error_std = df_15m['forecast_error'].std()
print(f"Standard Deviation of 15m Forecast Error: {error_std:.4f}")

# Threshold = 2 Standard Deviations
threshold_pos = 2 * error_std

# Flag Shocks
df_15m['is_positive_shock'] = df_15m['forecast_error'] > threshold_pos
df_15m['is_absolute_shock'] = df_15m['forecast_error'].abs() > threshold_pos

print(f"Total Positive Shocks detected: {df_15m['is_positive_shock'].sum()}")



## 4. Phase Analysis: Did Intraday Shocks Increase During the Frenzy?

We assign our 3 regulatory phases to each 15-minute interval and calculate the percentage of intervals that experienced a shock.


In [ ]:

# Assign Regulatory Phases
conds_phase = [
    df_15m['date'] < '2023-09-04',
    (df_15m['date'] >= '2023-09-04') & (df_15m['date'] <= '2024-11-19'),
    df_15m['date'] >= '2024-11-20'
]
labels_phase = [
    'Phase 1: Pre-Frenzy',
    'Phase 2: Peak Frenzy',
    'Phase 3: Post-Mandate'
]
df_15m['phase'] = np.select(conds_phase, labels_phase, default='Other')

# Calculate Shock Frequencies
phase_summary = df_15m.groupby('phase').agg(
    total_15m_intervals=('log_ret_15m', 'count'),
    positive_shocks=('is_positive_shock', 'sum')
).reset_index()

phase_summary['positive_shock_freq_pct'] = (phase_summary['positive_shocks'] / phase_summary['total_15m_intervals']) * 100

display(phase_summary)



## 5. Visualizations


In [ ]:

# Resample to daily for the line plot to avoid plotting 30,000+ noisy points, 
# but keep the exact 15m shock points on top.
daily_avg = df_15m.groupby('date').agg({
    'forecast_vol': 'mean',
    'actual_vol': 'mean'
}).reset_index()

plt.figure(figsize=(16, 7))

# Plot Daily Averages for visual clarity
plt.plot(daily_avg['date'], daily_avg['forecast_vol'], color='#ff7f0e', linewidth=2, label='Forecasted Vol (Daily Avg)', zorder=2)
plt.plot(daily_avg['date'], daily_avg['actual_vol'], color='#a6cee3', linewidth=1, alpha=0.5, label='Actual Vol (Daily Avg)', zorder=1)

# Highlight Exact 15m Shocks
shocks_df = df_15m[df_15m['is_positive_shock']]
plt.scatter(shocks_df.index, shocks_df['actual_vol'], color='red', s=10, alpha=0.6, zorder=3, label=f'15m Positive Shocks (>2 Std Dev)')

# Phase lines
plt.axvline(pd.to_datetime('2023-09-04'), color='grey', linestyle='--', alpha=0.8)
plt.axvline(pd.to_datetime('2024-11-20'), color='red', linestyle='--', alpha=0.8)
plt.text(pd.to_datetime('2022-10-01'), plt.ylim()[1]*0.9, 'Phase 1\nPre-Frenzy', fontsize=12, color='black', alpha=0.7)
plt.text(pd.to_datetime('2024-02-01'), plt.ylim()[1]*0.9, 'Phase 2\nPeak Frenzy', fontsize=12, color='black', alpha=0.7)
plt.text(pd.to_datetime('2025-06-01'), plt.ylim()[1]*0.9, 'Phase 3\nPost-Mandate', fontsize=12, color='red', alpha=0.7)

plt.title('NIFTY 15m Volatility Shocks Overlaid on Daily Averages', fontsize=15, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Annualized Volatility', fontsize=12)
plt.legend(loc='upper right')
plt.grid(alpha=0.3)
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

sns.barplot(
    data=phase_summary, 
    x='phase', 
    y='positive_shock_freq_pct', 
    palette=['#1f77b4', '#d62728', '#2ca02c']
)

plt.title('Frequency of 15m Volatility Shocks by Regime (>2 Std Dev)', fontsize=14, fontweight='bold')
plt.xlabel('Regulatory Phase', fontsize=12)
plt.ylabel('Shock Frequency (%)', fontsize=12)

# Add value labels on top of bars
for i, v in enumerate(phase_summary['positive_shock_freq_pct']):
    plt.text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=12, fontweight='bold')

plt.show()
